# Model 2 — Plant Disease Detection (Kerala priority classes)

This notebook builds a CNN disease classifier using **transfer learning (MobileNetV2)**.

Datasets:
- Kaggle: PlantVillage (`emmarex/plantdisease`)
- Kaggle: Rice diseases (`minhhuy2810/rice-diseases-image-dataset`)

You should curate/keep only these Kerala-priority classes (folder names):
- **Rice**: `Leaf_Blast`, `Brown_Spot`, `Bacterial_Leaf_Blight`, `Sheath_Blight`, `Healthy`
- **Banana**: `Sigatoka`, `Panama_Wilt`, `Banana_Bract_Mosaic_Virus`, `Healthy`
- **Coconut**: `Bud_Rot`, `Leaf_Blight`, `Root_Wilt`, `Healthy`
- **Pepper**: `Phytophthora`, `Pollu_Beetle`, `Healthy`
- **General**: `Healthy`

Output: `disease_model.h5` + `class_names.json` and a `predict()` function with treatment suggestions.


In [ ]:
# Run this first in every notebook
!pip -q install tensorflow pandas numpy matplotlib seaborn scikit-learn
!pip -q install kaggle opencv-python Pillow

from google.colab import drive
drive.mount('/content/drive')

MODEL_SAVE_PATH = '/content/drive/MyDrive/krishi_mitra_models/'
DATA_PATH = '/content/drive/MyDrive/krishi_mitra_data/'

import os
from pathlib import Path
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)


## 1) Download datasets (Kaggle)

Upload `kaggle.json` to Colab, then run:

- `emmarex/plantdisease`
- `minhhuy2810/rice-diseases-image-dataset`

After downloading, **create a unified dataset** folder with this layout:

```
DATA_PATH/processed/disease_images/
  train/<class_name>/*.jpg
  val/<class_name>/*.jpg
  test/<class_name>/*.jpg
```

Use an 80/10/10 split.


In [ ]:
from pathlib import Path

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)

if Path('kaggle.json').exists():
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json

raw_dir = Path(DATA_PATH) / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d emmarex/plantdisease -p "{raw_dir}" -q
!kaggle datasets download -d minhhuy2810/rice-diseases-image-dataset -p "{raw_dir}" -q

print('Raw downloads in:', raw_dir)
print('Next: unzip & curate into processed/disease_images/ train/val/test folders')

## 2) Data loading + augmentation

We use `tf.keras.utils.image_dataset_from_directory` with built-in labels from folder names.

Augmentations: rotation, flip, zoom, brightness/contrast.


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

base_dir = Path(DATA_PATH) / 'processed' / 'disease_images'
train_dir = base_dir / 'train'
val_dir = base_dir / 'val'
test_dir = base_dir / 'test'

assert train_dir.exists(), f"Missing: {train_dir}"
assert val_dir.exists(), f"Missing: {val_dir}"
assert test_dir.exists(), f"Missing: {test_dir}"

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=42,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print('Classes:', num_classes)
print(class_names[:10])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomContrast(0.15),
])


## 3) Model architecture (MobileNetV2)

Phase 1: freeze base, train classification head.

Phase 2: unfreeze last ~30 layers for fine-tuning.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.layers.Rescaling(1./255)(inputs)
x = augment(x)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, min_lr=1e-6, monitor='val_loss'),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)


## 4) Fine-tuning (unfreeze last 30 layers)


In [ ]:
base_model.trainable = True

# Freeze most layers; fine-tune only the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)


## 5) Evaluation (accuracy + per-class metrics + confusion matrix)


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.title('Confusion Matrix')
plt.show()

## 6) Grad-CAM visualization

Grad-CAM helps explain which regions of the leaf influenced the prediction.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def show_gradcam_on_batch(ds, n=3):
    for images, labels in ds.take(1):
        images = images[:n]
        labels = labels[:n]
        probs = model.predict(images)
        preds = np.argmax(probs, axis=1)

        # MobileNetV2 last conv layer name typically 'Conv_1' or similar; confirm via model.summary()
        last_conv = None
        for layer in model.layers[::-1]:
            if isinstance(layer, tf.keras.layers.Conv2D):
                last_conv = layer.name
                break
        if last_conv is None:
            raise ValueError('No Conv2D layer found for Grad-CAM')

        for i in range(n):
            img = images[i:i+1]
            heatmap = make_gradcam_heatmap(img, model, last_conv)
            heatmap = np.uint8(255 * heatmap)

            plt.figure(figsize=(10, 4))
            plt.subplot(1, 3, 1)
            plt.imshow(images[i].numpy().astype('uint8'))
            plt.title(f"True: {class_names[int(labels[i])]}")
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(heatmap, cmap='jet')
            plt.title('Grad-CAM heatmap')
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(images[i].numpy().astype('uint8'))
            plt.imshow(heatmap, cmap='jet', alpha=0.35)
            plt.title(f"Pred: {class_names[int(preds[i])]} ({float(np.max(probs[i])):.2f})")
            plt.axis('off')
            plt.show()

show_gradcam_on_batch(test_ds, n=3)


## 7) Save model + class names

Saves to:
- local project path: `ml_models/2_disease_detector/model/`
- Drive folder: `MODEL_SAVE_PATH/disease_detector/`


In [ ]:
import json
from pathlib import Path

local_dir = Path('ml_models/2_disease_detector/model')
local_dir.mkdir(parents=True, exist_ok=True)
model.save(local_dir / 'disease_model.h5')
with open(local_dir / 'class_names.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, indent=2)

drive_dir = Path(MODEL_SAVE_PATH) / 'disease_detector'
drive_dir.mkdir(parents=True, exist_ok=True)
model.save(drive_dir / 'disease_model.h5')
with open(drive_dir / 'class_names.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, indent=2)

print('Saved to local:', local_dir)
print('Saved to drive:', drive_dir)


## 8) `predict()` function (treatment-aware)

This is a practical inference helper that:
- loads an image
- predicts disease + confidence
- assigns a severity band
- provides Kerala-friendly organic/chemical/preventive suggestions


In [ ]:
import numpy as np
from PIL import Image

TREATMENTS = {
    'Leaf_Blast': {
        'organic': ['Use Trichoderma-enriched compost', 'Maintain proper spacing for airflow'],
        'chemical': [{'name': 'Tricyclazole', 'dosage': '0.6 g/L', 'frequency': '2 sprays, 10 days apart'}],
        'preventive': ['Use resistant varieties where possible', 'Avoid excess nitrogen'],
        'crop': 'Rice'
    },
    'Brown_Spot': {
        'organic': ['Balanced nutrition; add potash if deficient'],
        'chemical': [{'name': 'Mancozeb', 'dosage': '2 g/L', 'frequency': '2 sprays, 10 days apart'}],
        'preventive': ['Improve drainage', 'Use clean seed'],
        'crop': 'Rice'
    },
    'Bacterial_Leaf_Blight': {
        'organic': ['Avoid overhead irrigation', 'Remove heavily infected leaves'],
        'chemical': [{'name': 'Copper oxychloride', 'dosage': '2.5 g/L', 'frequency': '2 sprays, 7–10 days apart'}],
        'preventive': ['Use disease-free seedlings', 'Avoid dense planting'],
        'crop': 'Rice'
    },
    'Sheath_Blight': {
        'organic': ['Apply neem cake in soil', 'Improve field sanitation'],
        'chemical': [{'name': 'Validamycin', 'dosage': '2 ml/L', 'frequency': '2 sprays, 10 days apart'}],
        'preventive': ['Avoid excess nitrogen', 'Maintain water level properly'],
        'crop': 'Rice'
    },
    'Sigatoka': {
        'organic': ['Remove old leaves; improve ventilation'],
        'chemical': [{'name': 'Propiconazole', 'dosage': '1 ml/L', 'frequency': '2–3 sprays, 14 days apart'}],
        'preventive': ['Avoid waterlogging', 'Use clean planting material'],
        'crop': 'Banana'
    },
    'Panama_Wilt': {
        'organic': ['Use Trichoderma in pits', 'Destroy severely affected plants'],
        'chemical': [{'name': 'Soil drench (carbendazim)', 'dosage': '1 g/L', 'frequency': 'repeat after 15 days'}],
        'preventive': ['Use resistant varieties', 'Avoid moving soil from infected fields'],
        'crop': 'Banana'
    },
    'Bud_Rot': {
        'organic': ['Remove and burn infected tissues'],
        'chemical': [{'name': 'Bordeaux mixture', 'dosage': '1%', 'frequency': 'apply to crown; repeat after rains'}],
        'preventive': ['Improve drainage around palms', 'Avoid injuries to crown'],
        'crop': 'Coconut'
    },
    'Root_Wilt': {
        'organic': ['Apply compost + green manuring', 'Provide adequate irrigation in summer'],
        'chemical': [{'name': 'Micronutrient mix', 'dosage': 'as per label', 'frequency': '2–3 applications/year'}],
        'preventive': ['Integrated nutrient management', 'Control secondary pests'],
        'crop': 'Coconut'
    },
    'Phytophthora': {
        'organic': ['Improve drainage; avoid water stagnation'],
        'chemical': [{'name': 'Metalaxyl + Mancozeb', 'dosage': '2 g/L', 'frequency': '2 sprays, 10 days apart'}],
        'preventive': ['Use healthy cuttings', 'Mulch carefully; avoid collar wetness'],
        'crop': 'Pepper'
    },
}

def _severity(conf: float) -> str:
    if conf >= 0.90:
        return 'high'
    if conf >= 0.75:
        return 'medium'
    return 'low'

def predict(image_path=None, image_array=None):
    if image_array is None:
        if image_path is None:
            raise ValueError('Provide image_path or image_array')
        img = Image.open(image_path).convert('RGB').resize(IMG_SIZE)
        image_array = np.array(img)

    x = tf.convert_to_tensor(image_array, dtype=tf.float32)
    x = tf.image.resize(x, IMG_SIZE)
    x = tf.expand_dims(x, 0)

    prob = model.predict(x)[0]
    idx = int(np.argmax(prob))
    conf = float(prob[idx])
    disease = class_names[idx]

    info = TREATMENTS.get(disease, {
        'organic': ['Maintain field sanitation', 'Avoid overhead irrigation when possible'],
        'chemical': [{'name': 'Consult KAU/ICAR recommendation', 'dosage': '—', 'frequency': '—'}],
        'preventive': ['Use clean planting material', 'Scout weekly in monsoon'],
        'crop': 'Unknown'
    })

    should_consult = (conf < 0.65) or (disease.lower() == 'healthy')
    return {
        'disease': disease,
        'confidence': conf,
        'severity': _severity(conf),
        'affected_crop': info.get('crop', 'Unknown'),
        'treatment': {
            'organic': info['organic'],
            'chemical': info['chemical'],
            'preventive': info['preventive'],
        },
        'should_consult_expert': bool(should_consult),
    }

# Demo: run on any test image path you have
# result = predict(image_path=str(next(test_dir.rglob('*.jpg'))))
# result